# Tap Data and Station Data Merging
## By: William Griffith

Compress the raw dataset for sharing

In [ ]:
import requests #to get the url from the api
import numpy as np #to build arrays (sets of numbers)
import pandas as pd #dfs
import os as os #pull data from the folder
pd.set_option('display.max_rows', None) # so I can see all rows of a df
pd.set_option('display.max_columns', None)

Loading in Data

In [ ]:
df = pd.read_csv('Data/Raw_data/Boardings by Route Table_Full Data_data (2).csv.gz')
df["Date"] = pd.to_datetime(df["Date"])

C:\Users\griff\AppData\Local\Temp\ipykernel_10988\1213785295.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Date"] = pd.to_datetime(df["Date"])


Imports and settings

In [ ]:
import os
apikey = os.environ["WMATA_API_KEY"]


Date Range, and Raw Data

In [ ]:
print("Earliest date:", df["Date"].min())
print("Latest date:", df["Date"].max())
df.head(2)

Earliest date: 2022-12-20 00:00:00
Latest date: 2026-10-29 00:00:00


,Station Name,Time Period,Date,Day of Week,Holiday,Service Type,Year (Discrete),Year,Entries,NonTapped Entries,Paid Entries,Avg Daily Entries (copy),Avg Daily Entries Non-Rounded,Avg Daily Exits Rounded,Avg Daily NonTapped Entries,Avg Daily Tapped Entries,Data Range,DAY_TYPE,Exits,Hour,Month,Tableau X 98,Tableau Y 98,Tap Entries,Tap Exits,Avg Daily Entries
0,Ashburn,AM Peak (Open-9:30am),2026-05-30,Sat,No,Saturday,2026,2026,1,0,0,0.0K,0,0K,0,0,Showing Data Through 2026-05-30 00:00:00,Saturday,1,4,May,-167.932541,57.197032,1,1,0.0K
1,Innovation Center,AM Peak (Open-9:30am),2026-02-21,Sat,No,Saturday,2026,2026,0,0,0,0.0K,0,0K,0,0,Showing Data Through 2026-02-21 00:00:00,Saturday,1,4,February,-152.444212,41.546521,0,1,0.0K


Summing Entries by Station

In [ ]:
dftot = df.groupby('Station Name')[['Tap Entries','NonTapped Entries','Entries']].sum()
dftot.rename(columns={'Tap Entries':'Total_Tap_Entries','NonTapped Entries':'Total_NonTapped_Entries','Entries':'Total_Entries'}, inplace=True)
dftot.head(5)

,Total_Tap_Entries,Total_NonTapped_Entries,Total_Entries
Station Name,,,
Addison Rd,1713710,237579,1951289
Anacostia,4711541,965328,5676869
Archives,6161251,318525,6479776
Arlington Cemetery,1365615,57273,1422888
Ashburn,1960332,77441,2037773


Calculating the amount of collectionDays
- (As there can be multiple collections on one Day)

In [ ]:
df["justDate"] = df["Date"].dt.date #create a new column that is just the date, not the time
dfUnique = df.drop_duplicates(subset=['Station Name','justDate']) #drop_duplicates on entires that share both 
dfStationTot=dfUnique['Station Name'].value_counts()
dfStationTot = dfStationTot.reset_index().rename(columns={'count': 'collectionDays'}) #reset the index and name the day count
dfStationTot.head(5) #print the first 5 rows of the dfStationTot

,Station Name,collectionDays
0,Ashburn,1410
1,Innovation Center,1410
2,Spring Hill,1410
3,Rockville,1410
4,Court House,1410


Calculating the Average Daily Entry Statistics

- Also, 'Notappercent' =  100*('AvgDaily_NonTapped_Entries'/'AvgDaily_Entries')

In [ ]:
dfAgg = pd.merge(dftot, dfStationTot, on='Station Name')
dfAgg[['AvgDaily_Entries', 'AvgDaily_NonTapped_Entries', 'AvgDaily_Tap_Entries']] = dfAgg[['Total_Entries', 'Total_NonTapped_Entries', 'Total_Tap_Entries']].div(dfAgg['collectionDays'], axis=0)
dfAgg['NoTap_percent'] = dfAgg['AvgDaily_NonTapped_Entries'] / dfAgg['AvgDaily_Entries'] * 100
dfAvg = dfAgg[['Station Name', 'AvgDaily_Tap_Entries', 'AvgDaily_NonTapped_Entries', 'AvgDaily_Entries', 'NoTap_percent', 'collectionDays']].copy()
dfAvg.head(5)


,Station Name,AvgDaily_Tap_Entries,AvgDaily_NonTapped_Entries,AvgDaily_Entries,NoTap_percent,collectionDays
0,Addison Rd,1215.397163,168.495745,1383.892908,12.175490,1410
1,Anacostia,3341.518440,684.629787,4026.148227,17.004585,1410
2,Archives,4369.681560,225.904255,4595.585816,4.915679,1410
3,Arlington Cemetery,968.521277,40.619149,1009.140426,4.025124,1410
4,Ashburn,1390.306383,54.922695,1445.229078,3.800276,1410


Loading in Station Location Data

In [ ]:
url = "https://api.wmata.com/Rail.svc/json/jStations?LineCode="  #url from wmata
params =  {"api_key": apikey,
            "Cache-Control": "no-cache" #idk wmata wants it
            }
response = requests.get(url, params=params) # 
data = response.json()
df = pd.DataFrame(data) 
stations = pd.json_normalize(df['Stations']) #normalizing data
stations.sort_values(by='Name', inplace=True) #sorting by name
stations.head(5)

,Code,Name,StationTogether1,StationTogether2,LineCode1,LineCode2,LineCode3,LineCode4,Lat,Lon,Address.Street,Address.City,Address.State,Address.Zip
78,G03,Addison Road-Seat Pleasant,,,BL,SV,None,None,38.886713,-76.893592,100 Addison Rd S,Capitol Heights,MD,20743
70,F06,Anacostia,,,GR,None,None,None,38.862072,-76.995648,1101 Howard Rd SE,Washington,DC,20020
66,F02,Archives-Navy Memorial-Penn Quarter,,,GR,YL,None,None,38.893893,-77.021902,701 Pennsylvania Avenue NW,Washington,DC,20004
32,C06,Arlington Cemetery,,,BL,None,None,None,38.884574,-77.063108,1000 North Memorial Drive,Arlington,VA,22211
101,N12,Ashburn,,,SV,None,None,None,39.005283,-77.491537,43624 Croson Ln,Ashburn,VA,20148


Merging Data sets
- Now we have entry counts, and location Data

In [ ]:
dfAvg.loc[dfAvg['Station Name'] == 'Archives','Station Name'] = 'Archives-Navy Memorial-Penn Quarter'
dfAvg.loc[dfAvg['Station Name'] == 'U Street','Station Name'] = 'U Street/African-Amer Civil War Memorial/Cardozo'
dfAvg.loc[dfAvg['Station Name'] == 'Vienna','Station Name'] = 'Vienna/Fairfax-GMU'
dfAvg.loc[dfAvg['Station Name'] == 'Addison Rd','Station Name'] = 'Addison Road-Seat Pleasant'
    # Adding key
# Match the remaining ridership station names to the API names.
stationNameCorrections = {'Benning Rd': 'Benning Road', 'Branch Av': 'Branch Ave', 'Dulles Airport': 'Washington Dulles International Airport', 'Morgan Blvd': 'Morgan Boulevard', 'Naylor Rd': 'Naylor Road', 'Reagan National Airport': 'Ronald Reagan Washington National Airport', 'U St': 'U Street/African-Amer Civil War Memorial/Cardozo'}
dfAvg['Station Name'] = dfAvg['Station Name'].replace(stationNameCorrections)

dfAvg['key'] = dfAvg['Station Name'].str[:10].str.strip()
stations['key'] = stations['Name'].str[:10].str.strip()


dfAvg2 = pd.merge(dfAvg, stations, how='left', on='key').drop('key', axis=1).drop_duplicates(subset=['Station Name']).reset_index(drop=True) #merging the two dfs on the key and code columns, 
#also dropping duplicates based on 'Station Name' and resetting the index to avoid duplicate indices, 
#duplicates can occur if there are multiple stations with the same name but different lines. This ensures that each station is represented only once in the final merged dataframe.
#dfAvg2.loc[dfAvg2['Lat'].isnull()] #check for any stations still missing coordinates
dfAvg2.insert(0, 'Code', dfAvg2.pop('Code'))
dfAvg2.head(5)


,Code,Station Name,AvgDaily_Tap_Entries,AvgDaily_NonTapped_Entries,AvgDaily_Entries,NoTap_percent,collectionDays,Name,StationTogether1,StationTogether2,LineCode1,LineCode2,LineCode3,LineCode4,Lat,Lon,Address.Street,Address.City,Address.State,Address.Zip
0,G03,Addison Road-Seat Pleasant,1215.397163,168.495745,1383.892908,12.175490,1410,Addison Road-Seat Pleasant,,,BL,SV,None,None,38.886713,-76.893592,100 Addison Rd S,Capitol Heights,MD,20743
1,F06,Anacostia,3341.518440,684.629787,4026.148227,17.004585,1410,Anacostia,,,GR,None,None,None,38.862072,-76.995648,1101 Howard Rd SE,Washington,DC,20020
2,F02,Archives-Navy Memorial-Penn Quarter,4369.681560,225.904255,4595.585816,4.915679,1410,Archives-Navy Memorial-Penn Quarter,,,GR,YL,None,None,38.893893,-77.021902,701 Pennsylvania Avenue NW,Washington,DC,20004
3,C06,Arlington Cemetery,968.521277,40.619149,1009.140426,4.025124,1410,Arlington Cemetery,,,BL,None,None,None,38.884574,-77.063108,1000 North Memorial Drive,Arlington,VA,22211
4,N12,Ashburn,1390.306383,54.922695,1445.229078,3.800276,1410,Ashburn,,,SV,None,None,None,39.005283,-77.491537,43624 Croson Ln,Ashburn,VA,20148


Exporting Data Set

In [ ]:
dfAvg2.to_csv('Data/Output_data/merged_station_data.csv', index=False)